In [ ]:
from reserve import Reserve
from ia import IA
import random
########### Petite remarque sur les modifications que j'ai apportées ############
# J'ai fait une distinction entre les joueurs humains et l'IA uniquement dans les trois méthodes où l'IA doit faire un choix automatiquement : tour_joueur(), achat_cartes() et choisir_joueur()
# Pour cela, j'ai créé la classe IA et j'appelle directement les méthodes correspondantes de cette classe dans Partie
# J'ai également complété et modifié lancer_de() et tour_joueur() là où c'était nécessaire
# Et j'ai aussi ajouté au début du constructeur une vérification pour s'assurer que le nombre de joueurs est bien compris entre 2 et 4
#######################################################################################################################################

class Partie:
    def __init__(self, joueurs:list["Joueur"])->None:
        if len(joueurs) < 2 or len(joueurs) > 4:
            raise ValueError("Une partie doit comporter entre 2 et 4 joueurs")
        self._joueurs=joueurs
        self._reserve = Reserve()
        self._joueur_actuel = 0
    #prendre en compte : 1 tour par joueur avant que qqn ait "Gare", paiement avant revenu, 

    #ici on doit savoir si le joueur actuel est humain ou IA   
    def tour_joueur(self)->int:
        joueur = self._joueurs[self._joueur_actuel]

        #cas 1: IA
        if isinstance(joueur, IA):
            nombre_des = joueur.lancer_de() #appel lancer_de() de la classe IA

        #cas 2: humain
        else:
            if joueur.ville.monuments[0].construit: #s'il possede la gare
                nombre_des = int(input("Combien de dés voulez-vous lancer (1 ou 2) ? "))
                while nombre_des not in [1, 2]: #verifier que le choix est valide
                    nombre_des = int(input("Choix invalide. Choisissez 1 ou 2 : "))
            else:
                nombre_des = 1 
        #appeler lancer_de() de la classe Partie et lui donner nbr des a lancer et recuperer la somme
        resultat = self.lancer_de(nombre_des)
        return resultat

    def lancer_de(self, nombre_des: int) -> int: 
        resultat = 0
        for i in range(nombre_des): #somme 1 ou 2 des selon nb de des dans tour_joueur
            resultat += random.randint(1, 6)
        return resultat

    def joueur_suivant(self):
        pass
    
    def activer_carte(self, resultat):
        pass

    def paiement(self):
        pass

    def multiples_paiements(self): #sens antihoraire
        pass

    def revenu(self):
        pass

    def achat_cartes(self)-> bool:
    # au lieu de retourner la carte comme dans le jeu, ce que je propose c'est qu'on procède comme un achat
    # pas besoin de verifier la quantité des monuments pour les achats!!
        joueur = self._joueurs[self._joueur_actuel]

        #cas1: IA
        if isinstance(joueur, IA):
            nom = joueur.achat_cartes(self)  #on appelle achat_cartes() de la classe IA
            if nom is None:
                print("L'IA ne souhaite pas acheter de carte ou n'a pas assez d'argent")
                return False #l'achat n'a pas ete effectue

        #cas2: humain
        else:
            nom = input("Quelle carte voulez-vous acheter ? ")

        #cas1: Etablissemnt
        if nom in self._reserve._cartes: # verifie si c'est un établissement
            if not self._reserve.est_disponible(nom):
                print("Cette carte n'est plus disponible.")
                return False     
            
            carte = self._reserve._cartes[nom]["objet"] # ici j'ai recupere l'objet 
                                                       #pour pouvoir acceder au cout de la carte
            prix = carte.cout

            if joueur.argent >= prix:
                joueur.payer(prix)
                carte = self._reserve.prendre_carte(nom)
                joueur.ville.ajouter_carte(carte) 
                return True 
            
            print("Le joueur n'a pas assez d'argent")
            return False

        #cas2: monument
        else:
            for monument in joueur.ville.monuments:

                if monument.nom == nom:

                    if monument.construit:
                        print("Monument déjà construit")
                        return False

                    if joueur.argent >= monument.cout:
                        joueur.payer(monument.cout)
                        monument.construire()
                        return True

                    print("Pas assez d'argent")
                    return False

            print("Ce monument n'existe pas.")
            return False

        

    def choisir_joueur(self)->"Joueur": #petite correction pour le nom vu que ce nom etait deja defini dans la classe Chaine
        proprietaire = self._joueurs[self._joueur_actuel]

        #cas1: IA
        if isinstance(proprietaire, IA):
            return proprietaire.choisir_joueur(self) #on appelle choisir_joueur() de la classe IA retourne directement l'adversaire

        #cas2: humain
        else:
            adversaires = []

            for joueur in self._joueurs:
                if joueur != proprietaire:
                    adversaires.append(joueur) #on recupere la liste des adversaires

            for i in range(len(adversaires)):
                print(i + 1, adversaires[i].nom)#on affiche les adversaires numerotes avec leurs noms, i+1 juste pour commencer a compter a partir de 1
        
            choix = input("Choisissez un adversaire : ")
            while not choix.isdigit() or int(choix)<1 or int(choix)>len(adversaires):#.isdigit permet de verifier entier>0
                choix = input("Choix invalide. Réessayez : ")

            return adversaires[int(choix) - 1] #-1 car les indices commencent a 0 et int() car choix est str


    def victoire(self)->bool:
        joueur = self._joueurs[self._joueur_actuel]
        monuments = joueur.ville.monuments
        for monument in monuments:
            if monument.construit == False: #si on trouve un monument non construit False donc pas gagne
                return False

        return True #si tous les monuments sont construits